# FFT-accelerated beta-2 (L2) divergence experiments

> **Optional/legacy notebook.** The supported main experiments now use exact Gaussian formulas or direct spatial quadrature. This notebook retains the earlier periodic Fourier experiments and now contains only experiments based on the beta-2 divergence.

For the density-power-divergence convention, beta equal to 2 gives half the squared L2 distance:

$$
D_2(p,q)=\frac{1}{2}\int_{-\infty}^{\infty}(p(x)-q(x))^2\,\mathrm{d}x.
$$

For Gaussian KDEs based on the same sample, define the exact Gram matrix

$$
G_{ij}=\int \widehat f_{h_i}(x)\widehat f_{h_j}(x)\,\mathrm{d}x.
$$

Then

$$
D_2(\widehat f_{h_i},\widehat f_{h_j})
=\frac{1}{2}\left(G_{ii}+G_{jj}-2G_{ij}\right).
$$

The analytic Gaussian-KDE formula is exact but costs $O(N^2n^2)$. The experiments below instead evaluate the whole KDE family once by FFT and assemble $G$ with the matrix product $G \approx \Delta x\,FF^T$. The analytic helper is retained for the small validation check below. Control the FFT approximation with `beta2_half_width` and `beta2_spatial_points`.


In [25]:
from __future__ import annotations

import importlib

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kde_tools.bandwidths import (
    gaussian_mise_oracle_bandwidth,
)
from kde_tools.fourier import (
    beta2_divergence_matrix_fft,
    beta2_similarity_matrix_fft,
)
import kde_tools.similarities as kde_similarities

# Reload helpers that may have changed since this kernel was started.
importlib.reload(kde_similarities)
from kde_tools.similarities import (
    beta2_divergence_matrix,
    doob_stationary_mass,
    first_order_perron_vector,
    perron_vector,
    population_gaussian_beta2_divergence_matrix,
    toeplitz_corrected_choice,
)

np.set_printoptions(precision=5, suppress=True)
seed = 20260818


## 1. Configurable beta-2 selection histograms

Choose $n$, $N$, $M$, the bandwidth interval, and $\tau$ below. Every repetition uses the same sample for geometric and linear bandwidth grids and for the uncorrected and Toeplitz-corrected selectors. The four selection distributions are aggregated into common logarithmic bandwidth bins so that the grids can be compared on the same horizontal axis.


In [26]:
beta2_n = 200
beta2_N = 201
beta2_M = 100

beta2_h_min = 0.01
beta2_h_max = 2.0
beta2_tau = 1.0
beta2_seed = seed + 10
beta2_half_width = 12.0
beta2_spatial_points = 8192
beta2_histogram_bins = 40
beta2_validate_fft = True

if beta2_n < 1 or beta2_M < 1:
    raise ValueError("beta2_n and beta2_M must be positive.")
if beta2_N < 2:
    raise ValueError("beta2_N must be at least 2.")
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError("Require 0 < beta2_h_min < beta2_h_max.")
if beta2_tau <= 0.0:
    raise ValueError("beta2_tau must be positive.")
if beta2_half_width <= 0.0:
    raise ValueError("beta2_half_width must be positive.")
if beta2_spatial_points < 2 or beta2_spatial_points % 2:
    raise ValueError("beta2_spatial_points must be a positive even integer.")
if beta2_histogram_bins < 2:
    raise ValueError("At least two common histogram bins are required.")

beta2_geometric_grid = np.geomspace(
    beta2_h_min,
    beta2_h_max,
    beta2_N,
)
beta2_linear_grid = np.linspace(
    beta2_h_min,
    beta2_h_max,
    beta2_N,
)

beta2_histogram_styles = {
    "geometric_uncorrected": {
        "label": "geometric, uncorrected",
        "color": "#1f77b4",
        "width": 0.92,
        "opacity": 0.45,
    },
    "linear_uncorrected": {
        "label": "linear, uncorrected",
        "color": "#0b3c5d",
        "width": 0.72,
        "opacity": 0.68,
    },
    "geometric_corrected": {
        "label": "geometric, Toeplitz-corrected",
        "color": "#ff7f0e",
        "width": 0.52,
        "opacity": 0.76,
    },
    "linear_corrected": {
        "label": "linear, Toeplitz-corrected",
        "color": "#a84300",
        "width": 0.32,
        "opacity": 0.95,
    },
}


def beta2_select_indices(
    samples,
    bandwidth_grid,
    tau,
    half_width,
    spatial_points,
):
    similarity = beta2_similarity_matrix_fft(
        samples=samples,
        h_grid=bandwidth_grid,
        tau=tau,
        half_width=half_width,
        spatial_points=spatial_points,
    )
    uncorrected_index = int(
        np.argmax(doob_stationary_mass(similarity))
    )
    _, corrected_index, _ = toeplitz_corrected_choice(
        similarity=similarity,
        bandwidths=bandwidth_grid,
    )
    return uncorrected_index, corrected_index


print(
    f"beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, "
    f"tau={beta2_tau:g}"
)


beta=2, n=200, N=201, M=100, tau=1


In [27]:
if beta2_validate_fft:
    validation_rng = np.random.default_rng(beta2_seed)
    validation_samples = validation_rng.normal(
        size=min(beta2_n, 100)
    )
    for grid_label, bandwidth_grid in (
        ("geometric", beta2_geometric_grid),
        ("linear", beta2_linear_grid),
    ):
        validation_indices = np.unique(
            np.linspace(0, beta2_N - 1, min(beta2_N, 20), dtype=int)
        )
        validation_h_grid = bandwidth_grid[validation_indices]
        validation_exact = beta2_divergence_matrix(
            samples=validation_samples,
            h_grid=validation_h_grid,
        )
        validation_fft = beta2_divergence_matrix_fft(
            samples=validation_samples,
            h_grid=validation_h_grid,
            half_width=beta2_half_width,
            spatial_points=beta2_spatial_points,
        )
        maximum_error = float(
            np.max(np.abs(validation_fft - validation_exact))
        )
        print(
            f"FFT validation maximum absolute error ({grid_label}): "
            f"{maximum_error:.3e}"
        )

beta2_rng = np.random.default_rng(beta2_seed)
beta2_selected_indices = {
    key: np.empty(beta2_M, dtype=int)
    for key in beta2_histogram_styles
}

for repetition in range(beta2_M):
    current_samples = beta2_rng.normal(size=beta2_n)
    for grid_label, bandwidth_grid in (
        ("geometric", beta2_geometric_grid),
        ("linear", beta2_linear_grid),
    ):
        uncorrected_index, corrected_index = beta2_select_indices(
            samples=current_samples,
            bandwidth_grid=bandwidth_grid,
            tau=beta2_tau,
            half_width=beta2_half_width,
            spatial_points=beta2_spatial_points,
        )
        beta2_selected_indices[
            f"{grid_label}_uncorrected"
        ][repetition] = uncorrected_index
        beta2_selected_indices[
            f"{grid_label}_corrected"
        ][repetition] = corrected_index

    if (repetition + 1) % 5 == 0 or repetition + 1 == beta2_M:
        print(f"completed {repetition + 1}/{beta2_M} repetitions")

beta2_selected_bandwidths = {
    "geometric_uncorrected": beta2_geometric_grid[
        beta2_selected_indices["geometric_uncorrected"]
    ],
    "linear_uncorrected": beta2_linear_grid[
        beta2_selected_indices["linear_uncorrected"]
    ],
    "geometric_corrected": beta2_geometric_grid[
        beta2_selected_indices["geometric_corrected"]
    ],
    "linear_corrected": beta2_linear_grid[
        beta2_selected_indices["linear_corrected"]
    ],
}


FFT validation maximum absolute error (geometric): 2.176e-14
FFT validation maximum absolute error (linear): 2.315e-14
completed 5/100 repetitions
completed 10/100 repetitions
completed 15/100 repetitions
completed 20/100 repetitions
completed 25/100 repetitions
completed 30/100 repetitions
completed 35/100 repetitions
completed 40/100 repetitions
completed 45/100 repetitions
completed 50/100 repetitions
completed 55/100 repetitions
completed 60/100 repetitions
completed 65/100 repetitions
completed 70/100 repetitions
completed 75/100 repetitions
completed 80/100 repetitions
completed 85/100 repetitions
completed 90/100 repetitions
completed 95/100 repetitions
completed 100/100 repetitions


In [28]:
beta2_log_edges = np.linspace(
    np.log10(beta2_h_min),
    np.log10(beta2_h_max),
    beta2_histogram_bins + 1,
)
beta2_log_centers = 0.5 * (
    beta2_log_edges[:-1] + beta2_log_edges[1:]
)
beta2_log_width = beta2_log_edges[1] - beta2_log_edges[0]
beta2_histograms = {
    key: np.histogram(
        np.log10(selected_bandwidths),
        bins=beta2_log_edges,
    )[0]
    for key, selected_bandwidths in beta2_selected_bandwidths.items()
}

beta2_histogram_figure = go.Figure()
for key, style in beta2_histogram_styles.items():
    counts = beta2_histograms[key]
    nonzero_bins = np.flatnonzero(counts)
    beta2_histogram_figure.add_trace(
        go.Bar(
            x=beta2_log_centers[nonzero_bins],
            y=counts[nonzero_bins],
            customdata=10.0 ** beta2_log_centers[nonzero_bins],
            width=style["width"] * beta2_log_width,
            marker=dict(
                color=style["color"],
                line=dict(color="white", width=0.4),
            ),
            opacity=style["opacity"],
            name=style["label"],
            hovertemplate=(
                f"{style['label']}"
                "<br>bin center h=%{customdata:.6g}"
                "<br>frequency=%{y}<extra></extra>"
            ),
        )
    )

beta2_tick_positions = np.linspace(
    np.log10(beta2_h_min),
    np.log10(beta2_h_max),
    7,
)
beta2_tick_labels = [
    f"{10.0**position:.3g}" for position in beta2_tick_positions
]
beta2_histogram_figure.update_xaxes(
    title_text="selected bandwidth h (common logarithmic bins)",
    range=[np.log10(beta2_h_min), np.log10(beta2_h_max)],
    tickmode="array",
    tickvals=beta2_tick_positions,
    ticktext=beta2_tick_labels,
)
beta2_histogram_figure.update_yaxes(
    title_text=f"frequency among {beta2_M} repetitions",
    rangemode="tozero",
    dtick=1,
)
beta2_histogram_figure.update_layout(
    title=(
        f"Beta-2 selections by grid and correction: n={beta2_n}, "
        f"N={beta2_N}, M={beta2_M}, tau={beta2_tau:g}"
    ),
    width=1100,
    height=580,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18),
    margin=dict(t=100, b=130),
)
beta2_histogram_figure.show()


## 2. Beta-2 histograms for increasing sample size

This repeats the FFT-accelerated sweep for $n=100,200,\ldots,1000$ on fixed geometric and linear bandwidth grids with identical endpoints and numbers of candidates. Prefixes of the same maximum-size sample are reused for the smaller sample sizes, and every sample is used for all four grid/correction combinations. Each panel overlays the four distributions in common logarithmic bandwidth bins. Dashed vertical lines mark the geometric center of the bandwidth interval and the Gaussian MISE-oracle bandwidth.


In [29]:
beta2_n_sweep_values = np.arange(100, 1001, 100, dtype=int)
beta2_n_sweep_N = 201
beta2_n_sweep_M = 500
beta2_n_sweep_h_min = beta2_h_min
beta2_n_sweep_h_max = beta2_h_max
beta2_n_sweep_tau = beta2_tau
beta2_n_sweep_seed = beta2_seed + 2
beta2_n_sweep_histogram_bins = beta2_histogram_bins

beta2_n_sweep_geometric_grid = np.geomspace(
    beta2_n_sweep_h_min,
    beta2_n_sweep_h_max,
    beta2_n_sweep_N,
)
beta2_n_sweep_linear_grid = np.linspace(
    beta2_n_sweep_h_min,
    beta2_n_sweep_h_max,
    beta2_n_sweep_N,
)
beta2_n_sweep_center_h = np.sqrt(
    beta2_n_sweep_h_min * beta2_n_sweep_h_max
)
beta2_n_sweep_oracle_bandwidths = {
    int(current_n): gaussian_mise_oracle_bandwidth(int(current_n))
    for current_n in beta2_n_sweep_values
}

beta2_n_sweep_rng = np.random.default_rng(beta2_n_sweep_seed)
beta2_n_sweep_full_samples = beta2_n_sweep_rng.normal(
    size=(beta2_n_sweep_M, int(np.max(beta2_n_sweep_values)))
)
beta2_n_sweep_results = {}

for current_n in beta2_n_sweep_values:
    selected_indices = {
        key: np.empty(beta2_n_sweep_M, dtype=int)
        for key in beta2_histogram_styles
    }

    for repetition in range(beta2_n_sweep_M):
        current_samples = beta2_n_sweep_full_samples[
            repetition, : int(current_n)
        ]
        for grid_label, bandwidth_grid in (
            ("geometric", beta2_n_sweep_geometric_grid),
            ("linear", beta2_n_sweep_linear_grid),
        ):
            uncorrected_index, corrected_index = beta2_select_indices(
                samples=current_samples,
                bandwidth_grid=bandwidth_grid,
                tau=beta2_n_sweep_tau,
                half_width=beta2_half_width,
                spatial_points=beta2_spatial_points,
            )
            selected_indices[
                f"{grid_label}_uncorrected"
            ][repetition] = uncorrected_index
            selected_indices[
                f"{grid_label}_corrected"
            ][repetition] = corrected_index

    selected_bandwidths = {
        "geometric_uncorrected": beta2_n_sweep_geometric_grid[
            selected_indices["geometric_uncorrected"]
        ],
        "linear_uncorrected": beta2_n_sweep_linear_grid[
            selected_indices["linear_uncorrected"]
        ],
        "geometric_corrected": beta2_n_sweep_geometric_grid[
            selected_indices["geometric_corrected"]
        ],
        "linear_corrected": beta2_n_sweep_linear_grid[
            selected_indices["linear_corrected"]
        ],
    }
    beta2_n_sweep_results[int(current_n)] = {
        "selected_indices": {
            key: values.copy() for key, values in selected_indices.items()
        },
        "selected_bandwidths": selected_bandwidths,
        "oracle_bandwidth": beta2_n_sweep_oracle_bandwidths[int(current_n)],
    }
    print(
        f"completed n={current_n}: "
        f"geometric agreement="
        f"{np.mean(selected_indices['geometric_uncorrected'] == selected_indices['geometric_corrected']):.1%}, "
        f"linear agreement="
        f"{np.mean(selected_indices['linear_uncorrected'] == selected_indices['linear_corrected']):.1%}"
    )


completed n=100: geometric agreement=0.0%, linear agreement=0.0%
completed n=200: geometric agreement=0.0%, linear agreement=0.0%
completed n=300: geometric agreement=0.0%, linear agreement=0.0%
completed n=400: geometric agreement=0.0%, linear agreement=0.0%
completed n=500: geometric agreement=0.0%, linear agreement=0.0%
completed n=600: geometric agreement=0.0%, linear agreement=0.0%
completed n=700: geometric agreement=0.0%, linear agreement=0.0%
completed n=800: geometric agreement=0.0%, linear agreement=0.0%
completed n=900: geometric agreement=0.0%, linear agreement=0.0%
completed n=1000: geometric agreement=0.0%, linear agreement=0.0%


In [30]:
beta2_n_log_edges = np.linspace(
    np.log10(beta2_n_sweep_h_min),
    np.log10(beta2_n_sweep_h_max),
    beta2_n_sweep_histogram_bins + 1,
)
beta2_n_log_centers = 0.5 * (
    beta2_n_log_edges[:-1] + beta2_n_log_edges[1:]
)
beta2_n_log_width = beta2_n_log_edges[1] - beta2_n_log_edges[0]

for result in beta2_n_sweep_results.values():
    result["relative_histograms"] = {
        key: np.histogram(
            np.log10(selected_bandwidths),
            bins=beta2_n_log_edges,
        )[0] / beta2_n_sweep_M
        for key, selected_bandwidths in result[
            "selected_bandwidths"
        ].items()
    }

beta2_n_max_frequency = max(
    float(relative_frequencies.max())
    for result in beta2_n_sweep_results.values()
    for relative_frequencies in result["relative_histograms"].values()
)
beta2_n_line_height = 1.05 * beta2_n_max_frequency
beta2_n_rows = len(beta2_n_sweep_values)
beta2_n_stacked = make_subplots(
    rows=beta2_n_rows,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"n={value}" for value in beta2_n_sweep_values],
    vertical_spacing=0.012,
)

for row, current_n in enumerate(beta2_n_sweep_values, start=1):
    result = beta2_n_sweep_results[int(current_n)]
    for key, style in beta2_histogram_styles.items():
        relative_frequencies = result["relative_histograms"][key]
        nonzero_bins = np.flatnonzero(relative_frequencies)
        beta2_n_stacked.add_trace(
            go.Bar(
                x=beta2_n_log_centers[nonzero_bins],
                y=relative_frequencies[nonzero_bins],
                customdata=10.0 ** beta2_n_log_centers[nonzero_bins],
                width=style["width"] * beta2_n_log_width,
                marker=dict(
                    color=style["color"],
                    line=dict(color="white", width=0.3),
                ),
                opacity=style["opacity"],
                name=style["label"],
                legendgroup=key,
                hovertemplate=(
                    f"{style['label']}, n={current_n}"
                    "<br>bin center h=%{customdata:.6g}"
                    "<br>relative frequency=%{y:.1%}<extra></extra>"
                ),
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )

    for marker_h, name, color, legend_group in (
        (
            beta2_n_sweep_center_h,
            "geometric center",
            "#555555",
            "beta2-center",
        ),
        (
            result["oracle_bandwidth"],
            "Gaussian MISE oracle",
            "#d62728",
            "beta2-oracle",
        ),
    ):
        marker_x = np.log10(marker_h)
        beta2_n_stacked.add_trace(
            go.Scatter(
                x=[marker_x, marker_x],
                y=[0.0, beta2_n_line_height],
                mode="lines",
                line=dict(color=color, width=2, dash="dash"),
                name=name,
                legendgroup=legend_group,
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )
    beta2_n_stacked.update_yaxes(
        range=[0.0, beta2_n_line_height],
        tickformat=".0%",
        nticks=4,
        row=row,
        col=1,
    )

beta2_n_tick_positions = np.linspace(
    np.log10(beta2_n_sweep_h_min),
    np.log10(beta2_n_sweep_h_max),
    7,
)
beta2_n_tick_labels = [
    f"{10.0**position:.3g}" for position in beta2_n_tick_positions
]
beta2_n_stacked.update_xaxes(
    range=[
        np.log10(beta2_n_sweep_h_min),
        np.log10(beta2_n_sweep_h_max),
    ],
    tickmode="array",
    tickvals=beta2_n_tick_positions,
    ticktext=beta2_n_tick_labels,
)
beta2_n_stacked.update_xaxes(
    title_text="selected bandwidth h (common logarithmic bins)",
    row=beta2_n_rows,
    col=1,
)
beta2_n_stacked.update_layout(
    title=(
        "FFT beta-2 selections for increasing n: "
        f"geometric/linear and corrected/uncorrected, "
        f"N={beta2_n_sweep_N}, M={beta2_n_sweep_M}"
    ),
    width=1100,
    height=230 * beta2_n_rows + 190,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.025),
    margin=dict(t=100, b=180),
)
beta2_n_stacked.show()


## 3. Sample, exact-population, and first-order Perron vectors

This experiment compares three sum-normalized Perron vectors on the same bandwidth grid:

1. the sample vector obtained from the FFT beta-2 similarity matrix;
2. the exact population vector obtained by assembling the analytic Gaussian population matrix and solving its Perron eigenproblem numerically; and
3. the first-order perturbation formula

$$
v_i^{(1)}
=\frac{1}{N}
-\frac{\tau}{N^2}\left(r_i-\bar r\right),
\qquad
r_i=\sum_j D_{ij}^{\mathrm{pop}}.
$$

The implicit row-sum characterization determines the maximum of the first-order vector; it does not by itself provide the full exact Perron vector. All curves are displayed as $Nv_i$, so the horizontal value one corresponds to the uniform vector.

In [11]:
perron_comparison_n = 1000
perron_comparison_N = 191
perron_comparison_h_min = 0.01
perron_comparison_h_max = 2.0
perron_comparison_tau = 1.0
# Match the first repetition of the beta-2 sample-size sweep.
perron_comparison_seed = seed + 12
perron_comparison_half_width = beta2_half_width
perron_comparison_spatial_points = beta2_spatial_points

if perron_comparison_n < 1 or perron_comparison_N < 2:
    raise ValueError("Require n >= 1 and N >= 2.")
if not 0.0 < perron_comparison_h_min < perron_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if perron_comparison_tau < 0.0:
    raise ValueError("perron_comparison_tau must be nonnegative.")

perron_comparison_h_grid = np.geomspace(
    perron_comparison_h_min,
    perron_comparison_h_max,
    perron_comparison_N,
)
perron_comparison_rng = np.random.default_rng(perron_comparison_seed)
perron_comparison_samples = perron_comparison_rng.normal(
    size=perron_comparison_n
)

perron_comparison_sample_similarity = beta2_similarity_matrix_fft(
    samples=perron_comparison_samples,
    h_grid=perron_comparison_h_grid,
    tau=perron_comparison_tau,
    half_width=perron_comparison_half_width,
    spatial_points=perron_comparison_spatial_points,
)
perron_comparison_sample_vector = perron_vector(
    perron_comparison_sample_similarity
)

perron_comparison_population_divergence = (
    population_gaussian_beta2_divergence_matrix(
        h_grid=perron_comparison_h_grid,
        target_sigma=1.0,
    )
)
perron_comparison_population_similarity = np.exp(
    -perron_comparison_tau
    * perron_comparison_population_divergence
)
np.fill_diagonal(perron_comparison_population_similarity, 1.0)
perron_comparison_population_vector = perron_vector(
    perron_comparison_population_similarity
)

perron_comparison_first_order_vector = first_order_perron_vector(
    divergence_matrix=perron_comparison_population_divergence,
    tau=perron_comparison_tau,
)
perron_comparison_row_sums = (
    perron_comparison_population_divergence.sum(axis=1)
)
perron_comparison_implicit_index = int(
    np.argmin(perron_comparison_row_sums)
)

perron_comparison_vectors = {
    "sample FFT": perron_comparison_sample_vector,
    "exact population": perron_comparison_population_vector,
    "first-order population": perron_comparison_first_order_vector,
}
perron_comparison_selected_indices = {
    name: int(np.argmax(vector))
    for name, vector in perron_comparison_vectors.items()
}

for name, selected_index in perron_comparison_selected_indices.items():
    print(
        f"{name}: index={selected_index}, "
        f"h={perron_comparison_h_grid[selected_index]:.6g}"
    )
print(
    "implicit row-sum characterization: "
    f"index={perron_comparison_implicit_index}, "
    f"h={perron_comparison_h_grid[perron_comparison_implicit_index]:.6g}"
)
print(
    "maximum population first-order error: "
    f"{np.max(np.abs(perron_comparison_population_vector - perron_comparison_first_order_vector)):.3e}"
)

sample FFT: index=137, h=0.456209
exact population: index=138, h=0.46911
first-order population: index=138, h=0.46911
implicit row-sum characterization: index=138, h=0.46911
maximum population first-order error: 1.577e-06


In [12]:
perron_comparison_colors = {
    "sample FFT": "#1f77b4",
    "exact population": "#d62728",
    "first-order population": "#2ca02c",
}
perron_comparison_dashes = {
    "sample FFT": "solid",
    "exact population": "solid",
    "first-order population": "dash",
}

perron_comparison_figure = go.Figure()
for name, vector in perron_comparison_vectors.items():
    selected_index = perron_comparison_selected_indices[name]
    relative_vector = perron_comparison_N * vector
    perron_comparison_figure.add_trace(
        go.Scatter(
            x=perron_comparison_h_grid,
            y=relative_vector,
            customdata=vector,
            mode="lines",
            line=dict(
                color=perron_comparison_colors[name],
                width=2.5,
                dash=perron_comparison_dashes[name],
            ),
            name=name,
            hovertemplate=(
                f"{name}"
                "<br>h=%{x:.6g}"
                "<br>N v(h)=%{y:.6g}"
                "<br>v(h)=%{customdata:.8f}<extra></extra>"
            ),
        )
    )
    perron_comparison_figure.add_trace(
        go.Scatter(
            x=[perron_comparison_h_grid[selected_index]],
            y=[relative_vector[selected_index]],
            mode="markers",
            marker=dict(
                color=perron_comparison_colors[name],
                size=10,
                symbol="diamond",
                line=dict(color="white", width=1),
            ),
            name=f"{name} maximum",
            showlegend=False,
            hovertemplate=(
                f"{name} maximum"
                f"<br>index={selected_index}"
                "<br>h=%{x:.6g}"
                "<br>N v(h)=%{y:.6g}<extra></extra>"
            ),
        )
    )

perron_comparison_figure.add_hline(
    y=1.0,
    line=dict(color="#777777", width=1.5, dash="dot"),
    annotation_text="uniform vector",
    annotation_position="bottom right",
)
perron_comparison_figure.update_xaxes(
    title_text="bandwidth h (logarithmic scale)",
    type="log",
    range=[
        np.log10(perron_comparison_h_min),
        np.log10(perron_comparison_h_max),
    ],
)
perron_comparison_figure.update_yaxes(
    title_text="relative Perron component N v(h) (uniform = 1)",
)
perron_comparison_figure.update_layout(
    title=(
        f"Sample, exact-population, and first-order beta-2 Perron vectors: "
        f"n={perron_comparison_n}, N={perron_comparison_N}, "
        f"tau={perron_comparison_tau:g}"
    ),
    width=1100,
    height=650,
    template="plotly_white",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.18,
    ),
    margin=dict(b=130),
)
perron_comparison_figure.show()

## 4. Selection distributions by correction and grid

This Monte Carlo experiment reuses each sample for geometric and linear bandwidth grids. For each grid it records both the largest component of the uncorrected Doob stationary distribution and the Toeplitz-corrected choice. All four distributions are shown together in common logarithmic bandwidth bins.


In [ ]:
correction_comparison_n = 1000
correction_comparison_N = 191
correction_comparison_M = 100
correction_comparison_h_min = 0.01
correction_comparison_h_max = 2.0
correction_comparison_tau = 1.0
correction_comparison_seed = seed + 30
correction_comparison_half_width = beta2_half_width
correction_comparison_spatial_points = beta2_spatial_points
correction_comparison_histogram_bins = beta2_histogram_bins

if correction_comparison_n < 1 or correction_comparison_M < 1:
    raise ValueError("n and M must be positive.")
if correction_comparison_N < 2:
    raise ValueError("N must be at least 2.")
if not 0.0 < correction_comparison_h_min < correction_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if correction_comparison_tau <= 0.0:
    raise ValueError("tau must be positive.")

correction_comparison_geometric_grid = np.geomspace(
    correction_comparison_h_min,
    correction_comparison_h_max,
    correction_comparison_N,
)
correction_comparison_linear_grid = np.linspace(
    correction_comparison_h_min,
    correction_comparison_h_max,
    correction_comparison_N,
)
correction_comparison_rng = np.random.default_rng(
    correction_comparison_seed
)
correction_comparison_indices = {
    key: np.empty(correction_comparison_M, dtype=int)
    for key in beta2_histogram_styles
}

for repetition in range(correction_comparison_M):
    current_samples = correction_comparison_rng.normal(
        size=correction_comparison_n
    )
    for grid_label, bandwidth_grid in (
        ("geometric", correction_comparison_geometric_grid),
        ("linear", correction_comparison_linear_grid),
    ):
        uncorrected_index, corrected_index = beta2_select_indices(
            samples=current_samples,
            bandwidth_grid=bandwidth_grid,
            tau=correction_comparison_tau,
            half_width=correction_comparison_half_width,
            spatial_points=correction_comparison_spatial_points,
        )
        correction_comparison_indices[
            f"{grid_label}_uncorrected"
        ][repetition] = uncorrected_index
        correction_comparison_indices[
            f"{grid_label}_corrected"
        ][repetition] = corrected_index

    if (repetition + 1) % 10 == 0:
        print(
            f"completed {repetition + 1}/"
            f"{correction_comparison_M} repetitions"
        )

correction_comparison_bandwidths = {
    "geometric_uncorrected": correction_comparison_geometric_grid[
        correction_comparison_indices["geometric_uncorrected"]
    ],
    "linear_uncorrected": correction_comparison_linear_grid[
        correction_comparison_indices["linear_uncorrected"]
    ],
    "geometric_corrected": correction_comparison_geometric_grid[
        correction_comparison_indices["geometric_corrected"]
    ],
    "linear_corrected": correction_comparison_linear_grid[
        correction_comparison_indices["linear_corrected"]
    ],
}

for key, selected_bandwidths in correction_comparison_bandwidths.items():
    style = beta2_histogram_styles[key]
    print(
        f"{style['label']}: mean h={selected_bandwidths.mean():.6g}, "
        f"median h={np.median(selected_bandwidths):.6g}"
    )

for grid_label in ("geometric", "linear"):
    agreement = np.mean(
        correction_comparison_indices[f"{grid_label}_uncorrected"]
        == correction_comparison_indices[f"{grid_label}_corrected"]
    )
    print(f"{grid_label} corrected/uncorrected agreement: {agreement:.1%}")


In [ ]:
correction_comparison_log_edges = np.linspace(
    np.log10(correction_comparison_h_min),
    np.log10(correction_comparison_h_max),
    correction_comparison_histogram_bins + 1,
)
correction_comparison_log_centers = 0.5 * (
    correction_comparison_log_edges[:-1]
    + correction_comparison_log_edges[1:]
)
correction_comparison_log_width = (
    correction_comparison_log_edges[1]
    - correction_comparison_log_edges[0]
)
correction_comparison_relative = {
    key: np.histogram(
        np.log10(selected_bandwidths),
        bins=correction_comparison_log_edges,
    )[0] / correction_comparison_M
    for key, selected_bandwidths in correction_comparison_bandwidths.items()
}

correction_comparison_figure = go.Figure()
for key, style in beta2_histogram_styles.items():
    relative_frequencies = correction_comparison_relative[key]
    nonzero_bins = np.flatnonzero(relative_frequencies)
    correction_comparison_figure.add_trace(
        go.Bar(
            x=correction_comparison_log_centers[nonzero_bins],
            y=relative_frequencies[nonzero_bins],
            customdata=(
                10.0 ** correction_comparison_log_centers[nonzero_bins]
            ),
            width=style["width"] * correction_comparison_log_width,
            marker=dict(
                color=style["color"],
                line=dict(color="white", width=0.4),
            ),
            opacity=style["opacity"],
            name=style["label"],
            hovertemplate=(
                f"{style['label']}"
                "<br>bin center h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
        )
    )

correction_comparison_tick_positions = np.linspace(
    np.log10(correction_comparison_h_min),
    np.log10(correction_comparison_h_max),
    7,
)
correction_comparison_tick_labels = [
    f"{10.0**position:.3g}"
    for position in correction_comparison_tick_positions
]
correction_comparison_figure.update_xaxes(
    title_text="selected bandwidth h (common logarithmic bins)",
    range=[
        np.log10(correction_comparison_h_min),
        np.log10(correction_comparison_h_max),
    ],
    tickmode="array",
    tickvals=correction_comparison_tick_positions,
    ticktext=correction_comparison_tick_labels,
)
correction_comparison_figure.update_yaxes(
    title_text="relative frequency",
    tickformat=".0%",
    rangemode="tozero",
)
correction_comparison_figure.update_layout(
    title=(
        f"FFT beta-2 selection distributions by grid and correction: "
        f"n={correction_comparison_n}, N={correction_comparison_N}, "
        f"M={correction_comparison_M}, tau={correction_comparison_tau:g}"
    ),
    width=1100,
    height=600,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18),
    margin=dict(t=110, b=130),
)
correction_comparison_figure.show()


## 5. Selections on geometric and linear grids

This experiment compares uncorrected and Toeplitz-corrected FFT beta-2 selections on two deterministic bandwidth grids with identical endpoints and the same number of candidates. The same Monte Carlo sample is used for all four combinations in each repetition.

The geometric grid assigns candidates uniformly in $\log h$, whereas the linear grid assigns candidates uniformly in $h$. Consequently, they represent different limiting measures over the bandwidth interval, and the selector can legitimately change. Since their grid indices do not represent the same bandwidths, all four selected-bandwidth distributions are aggregated into common logarithmic histogram bins.


In [ ]:
grid_comparison_n = 1000
grid_comparison_N = 191
grid_comparison_M = 100
grid_comparison_h_min = 0.01
grid_comparison_h_max = 2.0
grid_comparison_tau = 1.0
grid_comparison_seed = seed + 40
grid_comparison_half_width = beta2_half_width
grid_comparison_spatial_points = beta2_spatial_points
grid_comparison_histogram_bins = 40

if grid_comparison_n < 1 or grid_comparison_M < 1:
    raise ValueError("n and M must be positive.")
if grid_comparison_N < 2:
    raise ValueError("N must be at least 2.")
if not 0.0 < grid_comparison_h_min < grid_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if grid_comparison_tau <= 0.0:
    raise ValueError("tau must be positive.")
if grid_comparison_histogram_bins < 2:
    raise ValueError("At least two common histogram bins are required.")

grid_comparison_geometric_grid = np.geomspace(
    grid_comparison_h_min,
    grid_comparison_h_max,
    grid_comparison_N,
)
grid_comparison_linear_grid = np.linspace(
    grid_comparison_h_min,
    grid_comparison_h_max,
    grid_comparison_N,
)
grid_comparison_indices = {
    key: np.empty(grid_comparison_M, dtype=int)
    for key in beta2_histogram_styles
}
grid_comparison_rng = np.random.default_rng(grid_comparison_seed)

for repetition in range(grid_comparison_M):
    current_samples = grid_comparison_rng.normal(size=grid_comparison_n)
    for grid_label, bandwidth_grid in (
        ("geometric", grid_comparison_geometric_grid),
        ("linear", grid_comparison_linear_grid),
    ):
        uncorrected_index, corrected_index = beta2_select_indices(
            samples=current_samples,
            bandwidth_grid=bandwidth_grid,
            tau=grid_comparison_tau,
            half_width=grid_comparison_half_width,
            spatial_points=grid_comparison_spatial_points,
        )
        grid_comparison_indices[
            f"{grid_label}_uncorrected"
        ][repetition] = uncorrected_index
        grid_comparison_indices[
            f"{grid_label}_corrected"
        ][repetition] = corrected_index

    if (repetition + 1) % 10 == 0:
        print(
            f"completed {repetition + 1}/"
            f"{grid_comparison_M} repetitions"
        )

grid_comparison_bandwidths = {
    "geometric_uncorrected": grid_comparison_geometric_grid[
        grid_comparison_indices["geometric_uncorrected"]
    ],
    "linear_uncorrected": grid_comparison_linear_grid[
        grid_comparison_indices["linear_uncorrected"]
    ],
    "geometric_corrected": grid_comparison_geometric_grid[
        grid_comparison_indices["geometric_corrected"]
    ],
    "linear_corrected": grid_comparison_linear_grid[
        grid_comparison_indices["linear_corrected"]
    ],
}

for key, selected_bandwidths in grid_comparison_bandwidths.items():
    style = beta2_histogram_styles[key]
    print(
        f"{style['label']}: mean h={selected_bandwidths.mean():.6g}, "
        f"median h={np.median(selected_bandwidths):.6g}"
    )

for correction_label in ("uncorrected", "corrected"):
    absolute_differences = np.abs(
        grid_comparison_bandwidths[f"geometric_{correction_label}"]
        - grid_comparison_bandwidths[f"linear_{correction_label}"]
    )
    print(
        f"{correction_label} paired mean absolute grid difference: "
        f"{absolute_differences.mean():.6g}"
    )


In [ ]:
grid_comparison_log_edges = np.linspace(
    np.log10(grid_comparison_h_min),
    np.log10(grid_comparison_h_max),
    grid_comparison_histogram_bins + 1,
)
grid_comparison_log_centers = 0.5 * (
    grid_comparison_log_edges[:-1]
    + grid_comparison_log_edges[1:]
)
grid_comparison_log_width = (
    grid_comparison_log_edges[1] - grid_comparison_log_edges[0]
)
grid_comparison_relative = {
    key: np.histogram(
        np.log10(selected_bandwidths),
        bins=grid_comparison_log_edges,
    )[0] / grid_comparison_M
    for key, selected_bandwidths in grid_comparison_bandwidths.items()
}

grid_comparison_figure = go.Figure()
for key, style in beta2_histogram_styles.items():
    relative_frequencies = grid_comparison_relative[key]
    nonzero_bins = np.flatnonzero(relative_frequencies)
    grid_comparison_figure.add_trace(
        go.Bar(
            x=grid_comparison_log_centers[nonzero_bins],
            y=relative_frequencies[nonzero_bins],
            customdata=(
                10.0 ** grid_comparison_log_centers[nonzero_bins]
            ),
            width=style["width"] * grid_comparison_log_width,
            marker=dict(
                color=style["color"],
                line=dict(color="white", width=0.4),
            ),
            opacity=style["opacity"],
            name=style["label"],
            hovertemplate=(
                f"{style['label']}"
                "<br>bin center h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
        )
    )

grid_comparison_tick_positions = np.linspace(
    np.log10(grid_comparison_h_min),
    np.log10(grid_comparison_h_max),
    7,
)
grid_comparison_tick_labels = [
    f"{10.0**position:.3g}"
    for position in grid_comparison_tick_positions
]
grid_comparison_figure.update_xaxes(
    title_text="selected bandwidth h (common logarithmic bins)",
    range=[
        np.log10(grid_comparison_h_min),
        np.log10(grid_comparison_h_max),
    ],
    tickmode="array",
    tickvals=grid_comparison_tick_positions,
    ticktext=grid_comparison_tick_labels,
)
grid_comparison_figure.update_yaxes(
    title_text="relative frequency",
    tickformat=".0%",
    rangemode="tozero",
)
grid_comparison_figure.update_layout(
    title=(
        f"FFT beta-2 selections by grid and correction: "
        f"n={grid_comparison_n}, N={grid_comparison_N}, "
        f"M={grid_comparison_M}, tau={grid_comparison_tau:g}"
    ),
    width=1100,
    height=600,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18),
    margin=dict(t=110, b=130),
)
grid_comparison_figure.show()
